In [1]:
import sys
sys.path.insert(0, "../..")

from src.models        import build_logdeeponet
from src.data          import ODEIterableDataset, DirichletSampler, ConstrainedLHCSampler
from src.physics       import Robertson, ODEsolver
from src.training      import Trainer, build_dataloaders
from src.losses        import RobertsonPhysicsLoss

import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap

import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Initialize Roberston Model
k1 = 4e-2
k2 = 3e7
k3 = 1e4
system = Robertson([k1, k2, k3])

# Initialize Sampler Object 
sampler = ConstrainedLHCSampler(low=0.5, high=1.0)

# Initialize Solver
solver = ODEsolver(system).solve

In [3]:
t_span  = (1e-4, 1e6)

# Initialize Training and validation datasets
train_dataset    = ODEIterableDataset(size = 2000,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = t_span,
                                      log_sampling = True,
                                      method       = "BDF",
                                      output_mask  = None)

val_dataset      = ODEIterableDataset(size = 100,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = t_span,
                                      log_sampling = True,
                                      method       = "BDF", 
                                      output_mask  = None)

In [4]:
DEEPONET_CONFIG = {
    
    "hidden_size" : 256,
    "depth"       : 12,
    "latent_size" : 60,
    "input_size_b": 3,
    "input_size_t": 1,
    "output_size" : 3,
    "activation"  : "gelu",
    "t_span"      : t_span

}


# Initialize DeepONe network 

deeponet = build_logdeeponet(DEEPONET_CONFIG)

In [ ]:
TRAIN_CONFIG = {

    "num_epochs"     : 250,
    "learning_rate"  : 0.0001,
    "batch_size"     : 16,
    "num_workers"    : 6,
    "Save_model"     : True,
    "Save_directory" : "../../weights/best_robertson_6.pth"
}


optimizer = torch.optim.Adam

In [6]:
y_scale = [1, 3.6e-5, 1]

loss_fn = RobertsonPhysicsLoss(k1, k2, k3, y_scale, 
                                lambda_data=1.0, 
                                lambda_cons=1.0, 
                                lambda_ode=1e-2)

# Move to device along with your model
loss_fn = loss_fn.to(device)

In [7]:
# Initialize Train Object and Train 

trainer   = Trainer(
    model         = deeponet,
    train_dataset = train_dataset, 
    val_dataset   = val_dataset, 
    optimizer     = optimizer,
    loss_fn       = loss_fn, 
    train_cfg     = TRAIN_CONFIG,
    model_cfg     = DEEPONET_CONFIG,
    project       = "PINN_rober",
    mode          = "online" 
)

trainer.run()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/enricp/.netrc.
wandb: Currently logged in as: nikpursals to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training: 100%|██████████| 250/250 [15:37<00:00,  3.75s/it]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
